In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer,
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# ----------------------------------------------------
# Load Dataset (Alzheimer's Disease)
# ----------------------------------------------------
candidate_paths = [
    Path("Alzhimers.xlsx"),
    Path("Alzheimer.xlsx"),
    Path("clean_alzheimer.csv"),
    Path("../Other GANS/clean_alzheimer.csv"),
    Path("../../Other GANS/clean_alzheimer.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Alzheimer dataset not found. Place Alzhimers.xlsx or clean_alzheimer.csv in this folder."
    )

if data_path.suffix.lower() in {".xlsx", ".xls"}:
    raw_data = pd.read_excel(data_path)
else:
    raw_data = pd.read_csv(data_path)

target_col = "Group"
ad_data = raw_data.drop(columns=["Subject ID", "M/F", "MRI ID", "Hand"], errors="ignore")
ad_data[target_col] = ad_data[target_col].replace({"Demented": 1, "Nondemented": 0, "Converted": 1})
ad_data[target_col] = pd.to_numeric(ad_data[target_col], errors="coerce").fillna(0).astype(int)

for col in ad_data.select_dtypes(include=[np.number]).columns:
    if ad_data[col].isnull().any():
        ad_data[col] = ad_data[col].fillna(ad_data[col].mean())

for col in ad_data.select_dtypes(include=["object"]).columns:
    if ad_data[col].isnull().any():
        modes = ad_data[col].mode()
        ad_data[col] = ad_data[col].fillna(modes[0] if len(modes) else "")

# Use same variable name pattern as other notebooks
alzheimer_data = ad_data.copy()

X = alzheimer_data.drop(columns=[target_col])
y = alzheimer_data[target_col]

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(alzheimer_data)

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


In [3]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ---------------------------------------------------
# TRAIN / TEST SPLIT (NO LEAKAGE)
# ---------------------------------------------------

train_real, test_real = train_test_split(
    alzheimer_data,
    test_size=TEST_SIZE,
    stratify=alzheimer_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# CTABGAN
# ---------------------------------------------------

try:

    data_path = "alzheimer_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[],
        log_columns=[],
        mixed_columns={},
        integer_columns=[target_col],
        problem_type={"Classification": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    synthetic_ctabgan[target_col] = (
        pd.to_numeric(synthetic_ctabgan[target_col], errors="coerce")
        .round()
        .clip(0, 1)
        .astype(int)
    )

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)


================ SINGLE RUN ================


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
100%|██████████| 150/150 [08:51<00:00,  3.54s/it]


Finished training in 570.3841364383698  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 188.10it/s]|
Column Shapes Score: 62.15%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 318.37it/s]|
Column Pair Trends Score: 32.91%

Overall Score (Average): 47.53%

CTABGAN: 0.4753


In [4]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()
    data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, 1)
        .astype(int)
    )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 362.36it/s]|
Column Shapes Score: 73.22%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 348.34it/s]|
Column Pair Trends Score: 61.88%

Overall Score (Average): 67.55%

WGAN_GP: 0.6755


In [5]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_data[target_col] = (
            pd.to_numeric(synthetic_data[target_col], errors="coerce")
            .round()
            .clip(0, 1)
            .astype(int)
        )

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 458.21it/s]|
Column Shapes Score: 74.17%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 344.49it/s]|
Column Pair Trends Score: 58.37%

Overall Score (Average): 66.27%

CTGAN: 0.6627
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 603.24it/s]|
Column Shapes Score: 77.97%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 402.84it/s]|
Column Pair Trends Score: 61.62%

Overall Score (Average): 69.79%

CopulaGAN: 0.6979
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 863.74it/s]|
Column Shapes Score: 84.91%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 350.16it/s]|
Column Pair Trends Score: 76.14%

Overall Score (Average): 80.52%

TVAE: 0.8052
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 752.02it/s]|
Column Shapes 

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd

In [8]:
# TRTR (Train Real, Test Real)

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores)

    trtr_results.append({
        "Model": model_name,
        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)

--- Starting TRTR Evaluation (Train Real, Test Real) ---
Running LogReg...


Running SVM-RBF...
Running KNN...
Running NaiveBayes...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running AdaBoost...
Running MLP...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
0,LogReg,0.9560 ± 0.0215,0.9559 ± 0.0217,0.9583 ± 0.0199,0.9560 ± 0.0215
1,SVM-RBF,0.5680 ± 0.0359,0.5377 ± 0.0486,0.5932 ± 0.0433,0.5680 ± 0.0359
2,KNN,0.5333 ± 0.0556,0.5324 ± 0.0560,0.5336 ± 0.0556,0.5333 ± 0.0556
3,NaiveBayes,0.9547 ± 0.0171,0.9546 ± 0.0171,0.9563 ± 0.0161,0.9547 ± 0.0171
4,DecisionTree,0.9053 ± 0.0432,0.9051 ± 0.0435,0.9089 ± 0.0402,0.9053 ± 0.0432
5,RandomForest,0.9547 ± 0.0148,0.9546 ± 0.0149,0.9564 ± 0.0137,0.9547 ± 0.0148
6,ExtraTrees,0.9480 ± 0.0163,0.9479 ± 0.0164,0.9499 ± 0.0146,0.9480 ± 0.0163
7,GradientBoost,0.9453 ± 0.0219,0.9452 ± 0.0219,0.9471 ± 0.0209,0.9453 ± 0.0219
8,AdaBoost,0.9493 ± 0.0259,0.9492 ± 0.0260,0.9516 ± 0.0238,0.9493 ± 0.0259
9,MLP,0.6173 ± 0.0745,0.5607 ± 0.1226,0.6930 ± 0.0954,0.6173 ± 0.0745


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51]
):

    results = []
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[label_col] = pd.to_numeric(train_df[label_col], errors="coerce").astype(int)
    test_df[label_col] = pd.to_numeric(test_df[label_col], errors="coerce").astype(int)

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=y_train
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=y_test
            )

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label=1,
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )

In [13]:
import pandas as pd

label_col = "Group"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=alzheimer_data,
    test_df=alzheimer_data,
    label="Group",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=alzheimer_data,
        label="Group",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.9560 ± 0.0224,0.9533 ± 0.0251,0.9886 ± 0.0139,0.9216 ± 0.0443
3,NaiveBayes,0.9547 ± 0.0171,0.9527 ± 0.0185,0.9776 ± 0.0163,0.9297 ± 0.0324
1,SVM-RBF,0.9547 ± 0.0232,0.9518 ± 0.0260,0.9886 ± 0.0140,0.9189 ± 0.0452
5,RandomForest,0.9533 ± 0.0137,0.9514 ± 0.0149,0.9752 ± 0.0190,0.9297 ± 0.0324
8,AdaBoost,0.9493 ± 0.0259,0.9471 ± 0.0281,0.9674 ± 0.0286,0.9297 ± 0.0501
6,ExtraTrees,0.9480 ± 0.0163,0.9457 ± 0.0181,0.9696 ± 0.0191,0.9243 ± 0.0378
7,GradientBoost,0.9453 ± 0.0219,0.9434 ± 0.0229,0.9621 ± 0.0315,0.9270 ± 0.0383
9,MLP,0.9387 ± 0.0281,0.9369 ± 0.0301,0.9485 ± 0.0325,0.9270 ± 0.0453
2,KNN,0.9160 ± 0.0292,0.9106 ± 0.0352,0.9513 ± 0.0446,0.8784 ± 0.0676
4,DecisionTree,0.9067 ± 0.0458,0.9092 ± 0.0424,0.8895 ± 0.0692,0.9324 ± 0.0277


CTGAN - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
3,NaiveBayes,0.6147 ± 0.0444,0.6580 ± 0.0489,0.5842 ± 0.0349,0.7568 ± 0.0855
0,LogReg,0.6120 ± 0.0345,0.6528 ± 0.0313,0.5849 ± 0.0293,0.7405 ± 0.0516
1,SVM-RBF,0.5853 ± 0.0452,0.5915 ± 0.0470,0.5761 ± 0.0426,0.6108 ± 0.0675
2,KNN,0.5773 ± 0.0473,0.5870 ± 0.0418,0.5707 ± 0.0480,0.6108 ± 0.0727
6,ExtraTrees,0.5760 ± 0.0519,0.5861 ± 0.0677,0.5626 ± 0.0482,0.6162 ± 0.1003
9,MLP,0.5587 ± 0.0604,0.5597 ± 0.0751,0.5499 ± 0.0649,0.5757 ± 0.1040
7,GradientBoost,0.5333 ± 0.0581,0.5666 ± 0.0631,0.5233 ± 0.0511,0.6243 ± 0.0985
5,RandomForest,0.5267 ± 0.0418,0.5683 ± 0.0435,0.5174 ± 0.0352,0.6351 ± 0.0757
8,AdaBoost,0.5187 ± 0.0475,0.5605 ± 0.0556,0.5108 ± 0.0408,0.6297 ± 0.1054
4,DecisionTree,0.4947 ± 0.1069,0.5185 ± 0.1000,0.4955 ± 0.1093,0.5514 ± 0.1102


CTGAN - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,0.9480 ± 0.0375,0.9457 ± 0.0388,0.9719 ± 0.0393,0.9216 ± 0.0459
0,LogReg,0.9320 ± 0.0173,0.9303 ± 0.0191,0.9376 ± 0.0194,0.9243 ± 0.0378
8,AdaBoost,0.9267 ± 0.0339,0.9260 ± 0.0318,0.9365 ± 0.0613,0.9189 ± 0.0342
1,SVM-RBF,0.9120 ± 0.0275,0.9080 ± 0.0302,0.9321 ± 0.0216,0.8865 ± 0.0510
3,NaiveBayes,0.9013 ± 0.0369,0.8926 ± 0.0416,0.9578 ± 0.0400,0.8378 ± 0.0592
5,RandomForest,0.8787 ± 0.0340,0.8722 ± 0.0362,0.9101 ± 0.0523,0.8405 ± 0.0547
2,KNN,0.8573 ± 0.0426,0.8597 ± 0.0451,0.8317 ± 0.0389,0.8919 ± 0.0662
9,MLP,0.7973 ± 0.0487,0.8062 ± 0.0469,0.7639 ± 0.0481,0.8568 ± 0.0684
7,GradientBoost,0.7040 ± 0.0937,0.6971 ± 0.0624,0.7654 ± 0.1518,0.6892 ± 0.1447
4,DecisionTree,0.5480 ± 0.0889,0.5411 ± 0.0766,0.5561 ± 0.0985,0.5405 ± 0.0928


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,CTABGAN,LogReg,2.400000e-02,0.023008,0.051028,-0.002703,0.9560 ± 0.0224,0.9320 ± 0.0173
1,CTABGAN,NaiveBayes,5.333333e-02,0.060050,0.019790,0.091892,0.9547 ± 0.0171,0.9013 ± 0.0369
2,CTABGAN,SVM-RBF,4.266667e-02,0.043881,0.056437,0.032432,0.9547 ± 0.0232,0.9120 ± 0.0275
3,CTABGAN,RandomForest,7.466667e-02,0.079210,0.065094,0.089189,0.9533 ± 0.0137,0.8787 ± 0.0340
4,CTABGAN,AdaBoost,2.266667e-02,0.021167,0.030966,0.010811,0.9493 ± 0.0259,0.9267 ± 0.0339
5,CTABGAN,ExtraTrees,-2.220446e-16,0.000016,-0.002308,0.002703,0.9480 ± 0.0163,0.9480 ± 0.0375
6,CTABGAN,GradientBoost,2.413333e-01,0.246359,0.196689,0.237838,0.9453 ± 0.0219,0.7040 ± 0.0937
7,CTABGAN,MLP,1.413333e-01,0.130646,0.184584,0.070270,0.9387 ± 0.0281,0.7973 ± 0.0487
8,CTABGAN,KNN,5.866667e-02,0.050885,0.119562,-0.013514,0.9160 ± 0.0292,0.8573 ± 0.0426
9,CTABGAN,DecisionTree,3.586667e-01,0.368069,0.333413,0.391892,0.9067 ± 0.0458,0.5480 ± 0.0889


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
5,WGAN_GP,0.062400,0.080992,-0.028881,0.158108
4,TVAE,0.070400,0.060864,0.125096,-0.009459
0,CTABGAN,0.101733,0.102329,0.105526,0.091081
3,GaussianCopula,0.118000,0.109429,0.166072,0.048919
1,CTGAN,0.382533,0.355320,0.414292,0.286757
2,CopulaGAN,0.399600,0.494617,0.397164,0.544324


In [14]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
